### Install And Import

In [ ]:
!pip install scikit-learn
!pip install tf_keras
!pip install pandas
!pip install numpy
!pip install seaborn
!pip install torch
!pip install sentencepiece
# !pip install tensorflow[and-cuda]




# Install required packages
!pip install -q transformers datasets
!pip install transformers
# Reinstall transformers and tensorflow to ensure proper setup for TF models
# !pip uninstall -y transformers
# !pip install -q transformers==4.41.0



# !pip uninstall -y tf_keras keras keras-nightly keras-preprocessing
# !pip install --upgrade "tensorflow>=2.12" transformers




!pip uninstall -y transformers
!pip install -q transformers==4.41.0

In [1]:
import os

import tensorflow as tf
import tf_keras

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import transformers


from transformers import __version__ as transformers_version

print(f"TensorFlow Version: {tf.__version__}")
print(f"Transformers Version: {transformers_version}")

I0000 00:00:1787723955.849880 2971108 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787723955.855467 2971108 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787723956.387280 2971108 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787723958.767135 2971108 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

TensorFlow Version: 2.21.0
Transformers Version: 4.41.0


### Data Processing

In [56]:
modelName = 'bert-base-uncased'

modelName = 'bert-base-uncased'
# modelName = 'bert-base-uncased'
# modelName = 'bert-base-uncased'



df = pd.read_csv('train_(1)_(2)_(1).csv')
test_df = pd.read_csv('test_(1)_(2)_(1).csv')

print(df.shape)

print(df.sample(n=5))

(44798, 4)
          ID          Review_Title                          Review  Rating
25289  27735             Wonderful  Great quality.cooling is nice!       1
41537  45553           Really Nice                              Ok       1
7289    7987  Utterly Disappointed                Spin not working       0
26422  28971               Awesome                    GOOD PRODUCT       1
13371  14676         Great product                          Superb       1


In [57]:
#Sentences and labels
sentences = df.Review.values
heading = df.Review_Title.values
labels = df.Rating.values


test_sentences = test_df.Review.values
test_heading = test_df.Review_Title.values

# test_labels = test_df.Rating.values

In [58]:
# sentences = df.apply(lambda x: x.Review_Title + "[SEP]" + x.Review, axis=1).values


# test_sentences = test_df.apply(lambda x: x.Review_Title + "[SEP]" + x.Review, axis=1).values

## Tokenize data using Bert Tokenizer

In [59]:
# from transformers import AutoTokenizer

tokenizer = transformers.AutoTokenizer.from_pretrained(modelName)


# It takes a single string and chops it into pieces (tokens) using BERT's specific vocabulary rules. (bert bas been trained on these tokens)
tokenized_texts_heading = [tokenizer.tokenize(word) for word in heading]
tokenized_texts = [tokenizer.tokenize(word) for word in sentences]

test_tokenized_texts_heading = [tokenizer.tokenize(word) for word in test_heading]
test_tokenized_texts = [tokenizer.tokenize(word) for word in test_sentences]



'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: c26636bc-3ea6-4869-85e7-848cefbf4ae9)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Token indices sequence length is longer than the specified maximum sequence length for this model (1197 > 512). Running this sequence through the model will result in indexing errors


In [ ]:


#We will use only first 512 tokens to do classification (this value can be changed)
max_length = 512
# tokenized_texts = [sent[:max_length] for sent in tokenized_texts]


for i in range(len(tokenized_texts)):
    start = 0
    if len(tokenized_texts[i]) + len(tokenized_texts_heading[i]) > max_length + 3:
        start = len(tokenized_texts[i]) + len(tokenized_texts_heading[i]) - max_length -3
    tokenized_texts[i] = ['[CLS]'] + tokenized_texts_heading[i] + ['[SEP]'] + tokenized_texts[i][start:] + ['[SEP]']

    # while len(tokenized_texts[i]) < max_length:
    #     tokenized_texts[i].append



# 
for i in range(len(test_tokenized_texts)):
    start = 0
    if len(test_tokenized_texts[i]) + len(test_tokenized_texts_heading[i]) > max_length + 3:
        start = len(test_tokenized_texts[i]) + len(test_tokenized_texts_heading[i]) - max_length -3
    test_tokenized_texts[i] = ['[CLS]'] + test_tokenized_texts_heading[i] + ['[SEP]'] + test_tokenized_texts[i][start:] + ['[SEP]']

    while len(test_tokenized_texts[i]) < max_length:
            test_tokenized_texts[i] = test_tokenized_texts[i] + ['PAD']

test_input_ids = [tokenizer.convert_tokens_to_ids(tok) for tok in test_tokenized_texts]




print(tokenized_texts[0])


In [ ]:
#80% data will be used for training while 20% will be used for test
trainX, testX, trainY, testY = train_test_split(tokenized_texts, labels, test_size=0.2)

In [ ]:
#Convert tokens into IDs
input_ids = [tokenizer.convert_tokens_to_ids(tok) for tok in tokenized_texts]


print(input_ids[0])


In [ ]:
# #Pad our tokens which might be less than max_length size
# input_ids = tf.keras.preprocessing.sequence.pad_sequences(input_ids,
#                                                           maxlen=max_length+2,
#                                                           truncating='post',
#                                                           padding='post')

Split data between training and test

Create Attention masks : Attention masks are useful to ignore padding tokens. Mask value will be set to 0 for padding tokens and 1 for actual tokens. We will create mask both for training and test data

In [ ]:
# # Create attention masks for training
# train_attn_masks = []

# # Create a mask of 1s for each token followed by 0s for padding
# for seq in trainX:
#   seq_mask = [float(i>0) for i in seq]
#   train_attn_masks.append(seq_mask)

In [ ]:
# # Create attention masks for Test
# test_attn_masks = []

# # Create a mask of 1s for each token followed by 0s for padding
# for seq in testX:
#   seq_mask = [float(i>0) for i in seq]
#   test_attn_masks.append(seq_mask)

In [ ]:
print(train_attn_masks[100])

### Build Model

In [ ]:
# from transformers import TFBertForSequenceClassification

model = transformers.TFBertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=10,
    from_pt=True   # explicitly convert from PyTorch
)

In [ ]:
model.summary()

In [ ]:
import tf_keras
# Use tf_keras optimizer to ensure compatibility with TFBert models in newer TF versions
optimizer = tf_keras.optimizers.Adam(learning_rate=3e-5, epsilon=1e-08, clipnorm=1.0)

In [ ]:
# Define loss and metrics using tf_keras
loss = tf_keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf_keras.metrics.SparseCategoricalAccuracy('accuracy')]

In [ ]:
# !pip install "tensorflow<2.16"

In [ ]:
# 2. Compile the model using tf_keras compatible objects
# This avoids the '_distribute_strategy' error in TF 2.16+
model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

### Train Model

In [ ]:
train_x_data = {'input_ids': np.array(trainX), 'attention_mask': np.array(train_attn_masks)}
test_x_data = {'input_ids': np.array(testX), 'attention_mask': np.array(test_attn_masks)}

In [ ]:
# Training the model with the prepared data
model.fit(
    train_x_data,
    trainY,
    validation_data=(test_x_data, testY),
    batch_size=16,
    epochs=10
)

In [ ]:
pass

In [ ]:
pass

In [ ]:
testData = pd.read_csv('test_(1)_(2)_(1).csv')

from here please

In [ ]:
# 1. Preprocess the test data (similar to training data)
test_sentences = testData.Review.values
test_tokenized = [tokenizer.tokenize(sent)[:max_length] for sent in test_sentences]
test_tokenized = [['[CLS]'] + sent + ['[SEP]'] for sent in test_tokenized]

# 2. Convert to IDs and Pad
test_input_ids = [tokenizer.convert_tokens_to_ids(sent) for sent in test_tokenized]
test_input_ids = tf.keras.preprocessing.sequence.pad_sequences(test_input_ids, maxlen=max_length+2, truncating='post', padding='post')

# 3. Create Attention Masks
test_masks = np.array([[float(i>0) for i in seq] for seq in test_input_ids])

# 4. Predict
predictions = model.predict({'input_ids': test_input_ids, 'attention_mask': test_masks})
print(predictions)

In [ ]:
# 1. Convert logits to class labels (0 or 1)
# We take the index of the highest score (argmax) for each prediction
predicted_labels = np.argmax(predictions.logits, axis=1)

# 2. Add predictions back to the test dataframe for easy viewing
testData['Predicted_Rating'] = predicted_labels

# 3. Display a sample of the results
print("Sample Predictions:")
display(testData[['Review', 'Predicted_Rating']].head(10))

In [ ]:
sol = testData[['ID', 'Predicted_Rating']]

In [ ]:
sol.to_csv("./soln.csv", index=False)

In [ ]:
from google.colab import files

files.download("./soln.csv")